In [ ]:
import numpy as np

RANDOM_SEED = 42

REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1','E2','E3'],
    'Hand':          ['E4','E5','E6','E7'],
    'Forearm':       ['E8','E9','E10','E11','E12','E13'],
    'Arm':           ['E14','E15','E16','E17','E18','E19','E20'],
}
ELECTRODE_TO_REGION = {e: r for r, es in REGION_TO_ELECTRODES.items() for e in es}
REGION_TO_LABEL     = {'Middle_Finger': 0, 'Hand': 1, 'Forearm': 2, 'Arm': 3}
LABEL_TO_REGION     = {v: k for k, v in REGION_TO_LABEL.items()}

In [ ]:
from pathlib import Path

BIDS_ROOT = Path("../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
subjects = ["sub-p0002"]
session, task, space = "ses-01", "task-S1Map", "MNI152NLin2009cAsym"
n_runs, HRF_DELAY, WINDOW = 4, 6.0, 1

RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "models").mkdir(parents=True, exist_ok=True)

In [ ]:
import json

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"TR: {tr} s")

In [ ]:
import pandas as pd

all_events = []
for run in range(1, n_runs + 1):
    path = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(path, sep='\t')
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)
print(f"Stimulation events: {len(stim_events)}")

In [ ]:
from nilearn.image import load_img, index_img, new_img_like, resample_to_img
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.maskers import NiftiMasker

first_run_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii"
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
mask_data = np.isin(atlas_img.get_fdata(), s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)
print(f"Voxels in S1 mask: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")

In [ ]:
X_list, y_list, run_labels = [], [], []

for run in range(1, n_runs + 1):
    bold_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii"
    bold_img = load_img(str(bold_path))
    run_evs = stim_events[stim_events['run'] == run]
    for _, event in run_evs.iterrows():
        vol_idx = int(np.round((event["onset"] + HRF_DELAY) / tr))
        vols = [vol_idx - WINDOW, vol_idx, vol_idx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vols):
            data = [masker.transform(index_img(bold_img, v)) for v in vols]
            data = [d[0] if len(d.shape) == 2 else d for d in data]
            X_list.append(np.mean(data, axis=0))
            y_list.append(event["region"])
            run_labels.append(run)

X = np.array(X_list)
y_encoded = np.array([REGION_TO_LABEL[r] for r in y_list])
run_labels = np.array(run_labels)
print(f"X: {X.shape}, y: {y_encoded.shape}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
print("Class weights:", {LABEL_TO_REGION[i]: f"{w:.3f}" for i, w in enumerate(class_weights)})

In [ ]:
import torch

mask_array = s1_mask_resampled.get_fdata()
voxel_coords = np.argwhere(mask_array > 0)
coord_to_idx = {tuple(c): i for i, c in enumerate(voxel_coords)}
num_nodes = len(voxel_coords)

edges_src, edges_dst = [], []
for i, (x, y, z) in enumerate(voxel_coords):
    for dx, dy, dz in [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]:
        nb = (x+dx, y+dy, z+dz)
        if nb in coord_to_idx:
            edges_src.append(i)
            edges_dst.append(coord_to_idx[nb])

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
torch.save(edge_index, RESULTS_DIR / "graph_edges.pt")
print(f"Nodes: {num_nodes}, Edges: {edge_index.shape[1]}, Avg degree: {edge_index.shape[1]/num_nodes:.2f}")

In [ ]:
import torch.nn.functional as F
from torch.nn import Module, Linear, BatchNorm1d
from torch_geometric.nn import GATConv, global_mean_pool

class SomatotopicGAT(Module):
    def __init__(self, hidden=32, heads=4, num_classes=4, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(1, hidden, heads=heads, concat=True, dropout=dropout)
        self.bn1 = BatchNorm1d(hidden * heads)
        self.conv2 = GATConv(hidden * heads, hidden, heads=heads, concat=True, dropout=dropout)
        self.bn2 = BatchNorm1d(hidden * heads)
        self.classifier = Linear(hidden * heads, num_classes)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.classifier(x)

n_params = sum(p.numel() for p in SomatotopicGAT().parameters())
print(f"Parameters: {n_params:,}")

In [ ]:
import copy
import torch.optim as optim
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

torch.manual_seed(RANDOM_SEED)

HIDDEN, HEADS, DROPOUT = 16, 2, 0.5
LR, WEIGHT_DECAY, PATIENCE, BATCH_SIZE = 0.001, 0.1, 100, 16

class_weights_tensor = torch.FloatTensor(class_weights)
criterion = CrossEntropyLoss(weight=class_weights_tensor)
fold_results = []

for test_run in range(1, n_runs + 1):
    print(f"\nFold {test_run}/4")
    train_mask = run_labels != test_run
    test_mask  = run_labels == test_run

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X[train_mask])
    X_test_s  = scaler.transform(X[test_mask])
    y_train, y_test = y_encoded[train_mask], y_encoded[test_mask]

    def make_dataset(X_s, y):
        return [Data(x=torch.FloatTensor(X_s[i].reshape(-1, 1)),
                     edge_index=edge_index,
                     y=torch.tensor(y[i], dtype=torch.long))
                for i in range(len(X_s))]

    train_loader = DataLoader(make_dataset(X_train_s, y_train), batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(make_dataset(X_test_s,  y_test),  batch_size=len(y_test))

    model = SomatotopicGAT(HIDDEN, HEADS, 4, DROPOUT)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_acc, no_improve, best_state = 0.0, 0, None

    for epoch in range(500):
        model.train()
        for batch in train_loader:
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                preds = model(batch.x, batch.edge_index, batch.batch).argmax(dim=1).numpy()
                acc = accuracy_score(y_test, preds)

        if epoch % 50 == 0:
            print(f"  epoch {epoch:3d} | best_acc={best_acc*100:.1f}%", end="\r")

        if acc > best_acc:
            best_acc, no_improve, best_state = acc, 0, copy.deepcopy(model.state_dict())
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  early stop at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        for batch in DataLoader(make_dataset(X_test_s, y_test), batch_size=len(y_test)):
            final_preds = model(batch.x, batch.edge_index, batch.batch).argmax(dim=1).numpy()

    fold_results.append({
        'fold': test_run, 'accuracy': accuracy_score(y_test, final_preds),
        'cm': confusion_matrix(y_test, final_preds),
        'model_state': best_state, 'scaler': scaler
    })
    print(f"  Acc: {fold_results[-1]['accuracy']*100:.2f}%")

mean_acc = np.mean([r['accuracy'] for r in fold_results])
std_acc  = np.std( [r['accuracy'] for r in fold_results])
print(f"\nMean LORO accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not fold_results:
    raise RuntimeError("fold_results is empty — re-run the training cell first.")

cm_total = np.sum([r['cm'] for r in fold_results], axis=0).astype(int)
labels = [LABEL_TO_REGION[i] for i in range(4)]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_total, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"GAT — {mean_acc*100:.1f}% ± {std_acc*100:.1f}%")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "gat_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
pd.DataFrame([{
    'fold': r['fold'], 'accuracy': r['accuracy'],
    'hidden': HIDDEN, 'heads': HEADS, 'dropout': DROPOUT,
    'lr': LR, 'weight_decay': WEIGHT_DECAY
} for r in fold_results]).to_csv(RESULTS_DIR / "gat_results.csv", index=False)

best_fold = max(fold_results, key=lambda r: r['accuracy'])
torch.save(best_fold['model_state'], RESULTS_DIR / "models" / "best_gat_model.pt")
print(f"Best fold: {best_fold['fold']} ({best_fold['accuracy']*100:.2f}%)")

In [ ]:
best_fold_data = max(fold_results, key=lambda r: r['accuracy'])
X_scaled_all = best_fold_data['scaler'].transform(X)

model_attn = SomatotopicGAT(HIDDEN, HEADS, 4, DROPOUT)
model_attn.load_state_dict(best_fold_data['model_state'])
model_attn.eval()

node_attn_per_class = np.zeros((4, num_nodes))

for c in range(4):
    attn_accum = np.zeros(num_nodes)
    indices = np.where(y_encoded == c)[0]
    with torch.no_grad():
        for i in indices:
            x_node = torch.FloatTensor(X_scaled_all[i].reshape(-1, 1))
            x1, (_, alpha1) = model_attn.conv1(x_node, edge_index, return_attention_weights=True)
            x1 = F.elu(model_attn.bn1(x1))
            _, (_, alpha2) = model_attn.conv2(x1, edge_index, return_attention_weights=True)
            node_attn = torch.zeros(num_nodes)
            node_attn.scatter_add_(0, edge_index[1], alpha2.mean(dim=-1))
            attn_accum += node_attn.numpy()
    node_attn_per_class[c] = attn_accum / len(indices)
    print(f"{LABEL_TO_REGION[c]}: done ({len(indices)} samples)")

np.save(RESULTS_DIR / "gat_node_attention.npy", node_attn_per_class)
print(f"Shape: {node_attn_per_class.shape}")

In [ ]:
from nilearn import datasets, plotting
from nilearn import surface as surf_utils

ATTN_DIR = FIGURES_DIR / "attention"
ATTN_DIR.mkdir(parents=True, exist_ok=True)
fsaverage = datasets.fetch_surf_fsaverage()

for c in range(4):
    class_name = LABEL_TO_REGION[c]
    attn_img = masker.inverse_transform(node_attn_per_class[c].reshape(1, -1))
    texture = surf_utils.vol_to_surf(attn_img, fsaverage.pial_left)
    nonzero = texture[np.abs(texture) > 0]
    vmax   = np.percentile(np.abs(nonzero), 95) if len(nonzero) > 0 else 1.0
    thresh = np.percentile(np.abs(nonzero), 10) if len(nonzero) > 0 else 0.0

    fig = plotting.plot_surf_stat_map(
        fsaverage.infl_left, stat_map=texture,
        hemi='left', view='lateral',
        threshold=thresh, vmax=vmax, colorbar=True,
        bg_on_data=True, alpha=0.7, cbar_tick_format='%.2e'
    )
    fig.savefig(str(ATTN_DIR / f"gat_attention_{class_name}.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {class_name}")

In [ ]:
from nilearn import plotting
from IPython.display import display

TOP_PERCENTILE = 25
winner_map = np.argmax(node_attn_per_class, axis=0)
max_attn = np.amax(node_attn_per_class, axis=0)
winner_values = np.where(max_attn >= np.percentile(max_attn, TOP_PERCENTILE), winner_map + 1, 0).astype(float)
winner_img = masker.inverse_transform(winner_values.reshape(1, -1))

view = plotting.view_img_on_surf(
    winner_img, surf_mesh='fsaverage',
    title="GAT Attention Map (1=Finger, 2=Hand, 3=Forearm, 4=Arm)",
    symmetric_cmap=False, vmin=1, vmax=4, threshold=1,
    vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"}
)
view.save_as_html(ATTN_DIR / "gat_attention_winner_take_all.html")
display(view)